# Analyse descriptive : retraits 2026, une ligne par opération, avec géolocalisation

**Objectif.** Comprendre en profondeur la donnée brute, sans aucun modèle : qualité, volumes, rythmes, montants, réseaux de cartes, retraits inaboutis, captures de carte et **dynamique à l'intérieur de la journée**. L'analyse doit faire apparaître d'elle-même les comportements atypiques des terminaux. Ces comportements serviront ensuite de cible à la détection.

**Hypothèses de comportements atypiques à confirmer ou infirmer par les données :**
1. **Silence anormal** : un terminal habituellement actif n'enregistre plus aucune opération pendant plusieurs heures (panne totale ou partielle).
2. **Rafale** : beaucoup d'opérations en très peu de temps, parfois du même montant.
3. **Retraits inaboutis en série** : retraits annulés, ou billets délivrés différents du montant demandé.
4. **Captures de carte anormales** : plusieurs cartes avalées le même jour ou en peu de temps.
5. **Afflux de cartes étrangères** ou de réseaux inhabituels pour ce terminal.

**Données.** Table brute de l'année 2026, déjà filtrée sur les trois libellés (retrait, autorisation-retrait, capture de carte), et jointe aux coordonnées géographiques corrigées des terminaux : site d'implantation, code postal, ville, longitude, latitude. Elle contient aussi les coupures réellement délivrées et la date de validité de la carte.

**Environnement.** Tout le calcul lourd se fait en **Spark**, sur le cluster. Seuls les résultats agrégés, petits, sont ramenés en pandas pour les graphiques. Le kernel doit être un environnement **distrib** (ex. `python36distrib_new`).

**Grain de l'analyse : la transaction.** Chaque opération garde sa ligne et reçoit ses propres indicateurs (réseau, capture réseau interne ou hors réseau interne, inabouti, délai depuis l'opération précédente, nombre d'opérations autour d'elle). Aucune journée n'est résumée en une seule ligne : les comptages par jour qui apparaissent dans les graphiques sont seulement des vues calculées à partir des transactions.

**Ne jamais faire `toPandas()` sur la table complète.**

## 0. Paramètres
Renseigner **soit** le nom du dataset dans le Flow (recommandé), **soit** le nom Hive complet `base.table`.

In [ ]:
DATASET_OPERATIONS = "NOM_DU_DATASET_OPERATIONS"   # nom affiché dans le Flow Dataiku
TABLE_HIVE_OPERATIONS = None                       # ex. "ma_base.ma_table" si lecture directe par Hive

DATE_DEBUT = None          # la table est déjà filtrée sur 2026 en amont ; ex. "2026-03-01" pour restreindre
DATE_FIN = None            # ex. "2026-09-30"
JOURS_INTRAJOURNEE = 400   # dynamique intra-journée (§ 11) sur les N derniers jours : 400 = toute l'année
DATASET_SORTIE_TRANSACTIONS = None   # ex. "retraits_2026_enrichis" : enregistre la table enrichie (§ 4 bis)
PERSISTER = True           # garde la table préparée en mémoire / disque du cluster (accélère les sections suivantes)
VERIFIER_DOUBLONS = True   # recherche de doublons exacts (coûteux sur 600 M de lignes)

# Noms des colonnes dans TA table. La détection est automatique ; corriger ici si une colonne est mal trouvée
# (le tableau de la section 2 indique, pour chaque colonne, ce qui a été retenu).
COLONNES = {
    "num_automate": "num_automate",
    "date_operation": "date_operation",
    "heure_operation": "heure_operation",
    "montant_ope": "montant_ope",
    "lib_ope": "lib_ope",
    "type_reseau": "type_reseau",
    "code_carte_privative": "code_carte_privative",
    "top_annule": "top_annule",
    "lib_motif_capture_carte": "lib_motif_capture_carte",
    "date_validite_carte": "date_validite_carte",
    "code_site": "code_site_implantation",
    "libelle_site": "libelle_site_implantation",
    "code_postal": "code_postal",
    "ville": "ville",
    "longitude": "longitude",
    "latitude": "latitude",
    "nb1": "nb1", "nb2": "nb2", "nb3": "nb3", "nb4": "nb4",
    "coupure1": "coupure1", "coupure2": "coupure2", "coupure3": "coupure3", "coupure4": "coupure4",
}
FORMAT_DATE = "ddMMyyyy"   # format de date_operation si c'est du texte (ex. "14032026") ; vérifier en section 2
DIVISEUR_MONTANT = 100     # montant_ope en centimes -> euros (mettre 1 si le montant est déjà en euros)
BILLETS_PAR_DEFAUT = {"nb1": 5, "nb2": 10, "nb3": 20, "nb4": 50}   # utilisé seulement si les coupures sont absentes

# Heures « de jour » pour l'étude des silences
HEURE_DEBUT_JOUR, HEURE_FIN_JOUR = 7, 22

## 1. Connexion Spark et lecture

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import dataiku
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
print("Spark", spark.version)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def lire_table(dataset, table_hive):
    """Lit la table par le dataset Dataiku (recommandé) ou par son nom Hive."""
    if dataset and dataset != "NOM_DU_DATASET_OPERATIONS":
        import dataiku.spark as dkuspark
        from pyspark.sql import SQLContext
        return dkuspark.get_dataframe(SQLContext(spark.sparkContext), dataiku.Dataset(dataset))
    if table_hive:
        return spark.table(table_hive)
    raise ValueError("Renseigner DATASET_OPERATIONS ou TABLE_HIVE_OPERATIONS dans la section 0.")


def en_pandas(sdf, max_lignes=500000):
    """Ramène un résultat Spark DÉJÀ AGRÉGÉ en pandas, avec garde-fou sur la taille."""
    pdf = sdf.limit(max_lignes).toPandas()
    if len(pdf) == max_lignes:
        print("Attention : résultat tronqué à {} lignes.".format(max_lignes))
    return pdf


def fmt(n):
    return "{:,.0f}".format(n).replace(",", " ")


source = lire_table(DATASET_OPERATIONS, TABLE_HIVE_OPERATIONS)
source.printSchema()

# Recherche de chaque colonne : nom indiqué, sinon nom sans tenir compte des majuscules, sinon mots-clés
MOTS_CLES = {
    "num_automate": [["automate"], ["terminal"]], "date_operation": [["date", "op"]], "heure_operation": [["heure"]],
    "montant_ope": [["montant"]], "lib_ope": [["lib", "ope"]], "type_reseau": [["type", "reseau"], ["reseau"]],
    "code_carte_privative": [["privative"]], "top_annule": [["annul"]], "lib_motif_capture_carte": [["motif"]],
    "date_validite_carte": [["validit"]], "code_site": [["code", "site"]], "libelle_site": [["lib", "site"]],
    "code_postal": [["postal"]], "ville": [["ville"]], "longitude": [["longitude"], ["lon"]],
    "latitude": [["latitude"], ["lat"]],
    "nb1": [["nb1"]], "nb2": [["nb2"]], "nb3": [["nb3"]], "nb4": [["nb4"]],
    "coupure1": [["coupure", "1"]], "coupure2": [["coupure", "2"]], "coupure3": [["coupure", "3"]],
    "coupure4": [["coupure", "4"]],
}
OBLIGATOIRES = ["num_automate", "date_operation", "montant_ope", "lib_ope"]
en_minuscules = {c.lower(): c for c in source.columns}
deja_pris = set()


def trouver(cle):
    nom = COLONNES.get(cle)
    if nom in source.columns:
        return nom
    if nom and nom.lower() in en_minuscules:
        return en_minuscules[nom.lower()]
    for mots in MOTS_CLES.get(cle, []):
        for c in source.columns:
            if c not in deja_pris and all(m in c.lower() for m in mots):
                return c
    return None


correspondance = {}
for cle in COLONNES:
    trouve = trouver(cle)
    correspondance[cle] = trouve
    if trouve:
        deja_pris.add(trouve)
display(pd.DataFrame([{"colonne attendue": k, "colonne retenue dans la table": v or "ABSENTE"}
                      for k, v in correspondance.items()]))
manquantes_obligatoires = [k for k in OBLIGATOIRES if not correspondance[k]]
if manquantes_obligatoires:
    raise ValueError("Colonnes indispensables introuvables : {} (corriger COLONNES en section 0)".format(
        manquantes_obligatoires))
non_utilisees = [c for c in source.columns if c not in deja_pris]
print("Colonnes de la table non utilisées :", non_utilisees or "aucune")

# Table de travail avec des noms de colonnes standard ; les colonnes absentes sont créées vides
brut = source.select([F.col(v).alias(k) if v else F.lit(None).cast("string").alias(k)
                      for k, v in correspondance.items()])
COUPURES_PRESENTES = all(correspondance["coupure{}".format(i)] for i in range(1, 5))
GEO_PRESENTE = bool(correspondance["longitude"] and correspondance["latitude"])

## 2. Premier contact : colonnes, exemples, volume

In [ ]:
print("Exemples de valeurs brutes (vérifier le format des dates, heures, montants, coupures, validité) :")
brut.show(5, truncate=False)
nb_lignes_brut = brut.count()
print("Nombre total de lignes :", fmt(nb_lignes_brut))

## 3. Qualité de la donnée brute
Valeurs manquantes, chaînes vides, formats invalides, valeurs prises par les colonnes catégorielles, doublons.

In [ ]:
# 3.1 Valeurs manquantes et chaînes vides, colonne par colonne (une seule passe)
aggs = []
for c in brut.columns:
    aggs.append(F.sum(F.col(c).isNull().cast("int")).alias(c + "__null"))
    aggs.append(F.sum((F.trim(F.col(c).cast("string")) == "").cast("int")).alias(c + "__vide"))
r = brut.agg(*aggs).collect()[0].asDict()
qualite = pd.DataFrame([{"colonne": c, "nb_null": r[c + "__null"], "nb_vide": r[c + "__vide"]} for c in brut.columns])
qualite["pct_null"] = (100 * qualite["nb_null"] / nb_lignes_brut).round(3)
qualite["pct_vide"] = (100 * qualite["nb_vide"] / nb_lignes_brut).round(3)
display(qualite.sort_values("pct_null", ascending=False))

In [ ]:
# 3.2 Valeurs prises par les colonnes catégorielles
for c in ["lib_ope", "type_reseau", "code_carte_privative", "top_annule", "lib_motif_capture_carte",
          "coupure1", "coupure2", "coupure3", "coupure4", "libelle_site", "ville"]:
    if correspondance.get(c):
        t = en_pandas(brut.groupBy(c).count().orderBy(F.desc("count")).limit(40))
        t["pct"] = (100 * t["count"] / nb_lignes_brut).round(3)
        print("\n=== {} : {} valeurs distinctes (40 premières) ===".format(c, brut.select(c).distinct().count()))
        display(t)

In [ ]:
# 3.3 Formats : dates, heures, montants
type_date = dict(brut.dtypes)["date_operation"]
if type_date in ("date", "timestamp"):
    date_op = F.to_date(F.col("date_operation"))
else:
    d_txt = F.trim(F.col("date_operation").cast("string"))
    if FORMAT_DATE in ("ddMMyyyy", "yyyyMMdd"):
        d_txt = F.lpad(d_txt, 8, "0")
    date_op = F.to_date(d_txt, FORMAT_DATE)
h_txt = F.lpad(F.regexp_replace(F.trim(F.col("heure_operation").cast("string")), "[^0-9]", ""), 6, "0")


def en_nombre(col):
    """Texte ou nombre -> double (virgule décimale acceptée)."""
    return F.regexp_replace(F.trim(F.col(col).cast("string")), ",", ".").cast("double")


controle = brut.select(
    date_op.alias("date"),
    h_txt.substr(1, 2).cast("int").alias("hh"),
    h_txt.substr(3, 2).cast("int").alias("mi"),
    F.col("montant_ope").cast("double").alias("montant_centimes"),
).agg(
    F.sum(F.col("date").isNull().cast("int")).alias("dates_invalides"),
    F.min("date").alias("date_min"),
    F.max("date").alias("date_max"),
    F.sum(((F.col("hh") > 23) | F.col("hh").isNull()).cast("int")).alias("heures_invalides"),
    F.sum(((F.col("mi") > 59) | F.col("mi").isNull()).cast("int")).alias("minutes_invalides"),
    F.sum(F.col("montant_centimes").isNull().cast("int")).alias("montants_non_numeriques"),
    F.sum((F.col("montant_centimes") <= 0).cast("int")).alias("montants_nuls_ou_negatifs"),
).collect()[0].asDict()
for k, v in controle.items():
    print("{:<28}: {}".format(k, v))
if controle["date_min"] is not None and (controle["date_min"].year < 2000 or controle["date_max"].year > 2030):
    print("ATTENTION : dates hors plage plausible, FORMAT_DATE est probablement faux (section 0).")
med = brut.select(F.expr("percentile_approx(cast(montant_ope as double), 0.5)").alias("m")).collect()[0]["m"]
print("Montant médian brut : {} -> {:.2f} € après division par {}".format(med, (med or 0) / DIVISEUR_MONTANT,
                                                                          DIVISEUR_MONTANT))

In [ ]:
# 3.4 Doublons exacts (toutes colonnes identiques)
if VERIFIER_DOUBLONS:
    nb_distinct = brut.distinct().count()
    print("Lignes en double exact : {} ({:.3f} %)".format(fmt(nb_lignes_brut - nb_distinct),
                                                         100 * (nb_lignes_brut - nb_distinct) / nb_lignes_brut))

## 4. Préparation : une ligne par opération, colonnes typées et enrichies
Reprend les règles de la table agrégée terminal-jour (montant en centimes, types d'opération, catégories de réseau, retraits inaboutis, motifs de capture), mais **sans agréger**.

In [ ]:
# --- Réseaux de cartes : mettre ici vos propres cartes ---
CODES_RESEAU = {}         # code de la colonne type_reseau -> nom du réseau
CODES_PRIVATIVE = {}      # code de la colonne code_carte_privative -> nom du réseau
GROUPES_RESEAU = {}       # nom du réseau -> groupe affiché dans les graphiques
RESEAUX_ETRANGERS = []    # noms des réseaux considérés comme étrangers
RESEAU_INTERNE = ""       # catégorie de réseau des cartes internes (à compléter)
MOTIFS = [
    ("OUBLI_OU_INCIDENT_LECTURE", ["OUBLI CARTE (OU INCIDENT LECTEUR CARTE)", "OUBLI DE CARTE",
                                   "INCIDENT LECTURE DE CARTE", "INCIDENT LECTURE CARTE"]),
    ("CODE_CONFIDENTIEL_DEPASSE", ["NOMBRE ESSAIS CODE CONFIDENTIEL DEPASSE", "NB ESSAI CODE CONFIDENTIEL DEPASSE",
                                   "NOMBRE ESSAI CODE DEPASSE"]),
    ("CARTE_PERDUE", ["CARTE PERDUE"]),
    ("CONSERVER_LA_CARTE", ["CONSERVER LA CARTE"]),
    ("CARTE_VOLEE", ["CARTE VOLEE", "CARTE VOLÉE", "CARTE DECLAREE VOLEE", "CARTE DÉCLARÉE VOLÉE", "VOL DE CARTE"]),
]

tr = F.lpad(F.trim(F.col("type_reseau").cast("string")), 2, "0")
cp = F.trim(F.col("code_carte_privative").cast("string"))
# Sans correspondance renseignée, le réseau garde son code brut
cat = F.when(F.lit(False), F.lit(""))
for code, nom in CODES_RESEAU.items():
    cat = cat.when(tr == code, nom)
for code, nom in CODES_PRIVATIVE.items():
    cat = cat.when(cp == code, nom)
cat = cat.when(cp.isNotNull() & (cp != ""), cp).otherwise(F.coalesce(tr, F.lit("inconnu")))

motif_txt = F.upper(F.trim(F.col("lib_motif_capture_carte")))
motif = None
for nom, valeurs in MOTIFS:
    motif = F.when(motif_txt.isin(valeurs), nom) if motif is None else motif.when(motif_txt.isin(valeurs), nom)
motif = motif.when(F.col("lib_motif_capture_carte").isNotNull(), "AUTRE_MOTIF")

# Montant délivré = somme des (nombre de billets x valeur de la coupure).
# Les coupures sont exprimées en euros ou en centimes : on le détecte sur leurs valeurs.
if COUPURES_PRESENTES:
    max_coupure = brut.select(F.max(F.greatest(*[en_nombre("coupure{}".format(i)) for i in range(1, 5)]))
                              .alias("m")).collect()[0]["m"] or 0
    diviseur_coupure = 100.0 if max_coupure > 500 else 1.0
    print("Coupures : valeur max = {} -> divisées par {:.0f}".format(max_coupure, diviseur_coupure))
    termes = [F.coalesce(en_nombre("nb{}".format(i)), F.lit(0.0)) *
              F.coalesce(en_nombre("coupure{}".format(i)), F.lit(0.0)) / diviseur_coupure for i in range(1, 5)]
else:
    print("Coupures absentes : valeurs par défaut", BILLETS_PAR_DEFAUT)
    termes = [F.coalesce(en_nombre(c), F.lit(0.0)) * v for c, v in BILLETS_PAR_DEFAUT.items()]
montant_billets = termes[0] + termes[1] + termes[2] + termes[3]

# Type d'opération, sans tenir compte des majuscules, accents de saisie ni espaces
lib = F.upper(F.trim(F.col("lib_ope").cast("string")))
type_op = (F.when(lib.like("%AUTORISATION%"), "AUTORISATION_RETRAIT")
           .when(lib.like("%CAPTURE%"), "CAPTURE")
           .when(lib.like("%RETRAIT%"), "RETRAIT"))
print("Libellés d'opération et type retenu :")
display(en_pandas(brut.groupBy("lib_ope", type_op.alias("type_op")).count().orderBy(F.desc("count"))))

if GROUPES_RESEAU:
    groupe = F.coalesce(F.create_map([F.lit(x) for kv in GROUPES_RESEAU.items() for x in kv])[F.col("cat_reseau")],
                        F.lit("Autres"))
else:
    groupe = F.col("cat_reseau")
etranger = F.col("cat_reseau").isin(RESEAUX_ETRANGERS) if RESEAUX_ETRANGERS else F.lit(False)

ops = (brut
       .filter(type_op.isNotNull())
       .select(
           F.col("num_automate"),
           date_op.alias("date"),
           h_txt.substr(1, 2).cast("int").alias("heure"),
           h_txt.substr(3, 2).cast("int").alias("minute"),
           F.to_timestamp(F.concat(F.date_format(date_op, "yyyyMMdd"), h_txt), "yyyyMMddHHmmss").alias("ts"),
           (en_nombre("montant_ope") / DIVISEUR_MONTANT).alias("montant"),
           F.trim(F.col("date_validite_carte").cast("string")).alias("validite_txt"),
           type_op.alias("type_op"),
           cat.alias("cat_reseau"),
           motif.alias("motif_capture"),
           F.coalesce(F.col("top_annule").cast("int"), F.lit(0)).alias("top_annule"),
           montant_billets.alias("montant_billets"),
       )
       .withColumn("est_retrait", F.col("type_op").isin("RETRAIT", "AUTORISATION_RETRAIT").cast("int"))
       .filter((F.col("type_op") == "CAPTURE") | (F.col("montant") > 0))
       .withColumn("groupe_reseau", groupe)
       .withColumn("est_etranger", etranger.cast("int"))
       .withColumn("ecart_billets", F.col("montant") - F.col("montant_billets"))
       .withColumn("inabouti_annule", ((F.col("est_retrait") == 1) & (F.col("top_annule") == 1)).cast("int"))
       .withColumn("inabouti_billets", ((F.col("est_retrait") == 1) & (F.abs(F.col("ecart_billets")) > 0.001)).cast("int"))
       .withColumn("inabouti", F.greatest("inabouti_annule", "inabouti_billets"))
       .withColumn("est_capture", (F.col("type_op") == "CAPTURE").cast("int"))
       .withColumn("est_reseau_interne", (F.col("cat_reseau") == RESEAU_INTERNE).cast("int"))
       .withColumn("est_capture_interne", (F.col("est_capture") * F.col("est_reseau_interne")).cast("int"))
       .withColumn("est_capture_hors_interne", (F.col("est_capture") * (1 - F.col("est_reseau_interne"))).cast("int"))
       .withColumn("est_nuit", ((F.col("heure") >= 22) | (F.col("heure") < 6)).cast("int"))
       .withColumn("jour_semaine", F.dayofweek("date"))          # 1 = dimanche ... 7 = samedi
       .withColumn("mois", F.trunc("date", "month")))
if DATE_DEBUT:
    ops = ops.filter(F.col("date") >= F.lit(DATE_DEBUT).cast("date"))
if DATE_FIN:
    ops = ops.filter(F.col("date") <= F.lit(DATE_FIN).cast("date"))
if PERSISTER:
    ops = ops.persist(StorageLevel.MEMORY_AND_DISK)

resume = ops.agg(F.count("*").alias("operations"), F.sum("est_retrait").alias("retraits"),
                 F.sum("est_capture").alias("captures"), F.countDistinct("num_automate").alias("terminal"),
                 F.min("date").alias("debut"), F.max("date").alias("fin"),
                 F.countDistinct("date").alias("jours")).collect()[0].asDict()
for k, v in resume.items():
    print("{:<12}: {}".format(k, fmt(v) if isinstance(v, (int, float)) else v))
DATE_MAX = resume["fin"]
print("Lignes écartées par le filtre (autres libellés, montants <= 0) : {}".format(fmt(nb_lignes_brut - resume["operations"])))

## 4 bis. Table enrichie : une ligne par transaction
Chaque opération reçoit des indicateurs calculés à partir de ses voisines sur le même terminal, triées dans le temps :

| Colonne | Signification |
|---|---|
| `delai_precedente_s` / `delai_suivante_s` | secondes depuis l'opération précédente / jusqu'à la suivante sur ce terminal |
| `rang_dans_journee` | numéro d'ordre de l'opération dans la journée du terminal |
| `nb_ope_5min` | nombre d'opérations du terminal dans les 5 minutes avant et après (rafale si élevé) |
| `meme_montant_precedent` | 1 si le montant est identique à celui de l'opération précédente |
| `inabouti_precedent` | 1 si l'opération précédente était aussi un retrait inabouti (série) |
| `capture_dans_l_heure` | nombre de captures sur ce terminal dans l'heure qui précède |

La table peut être enregistrée comme dataset (paramètre `DATASET_SORTIE_TRANSACTIONS`, dataset à créer d'abord dans le Flow) pour être réutilisée sans tout recalculer.

In [ ]:
w_ordre = Window.partitionBy("num_automate").orderBy("t")
w_jour_ordre = Window.partitionBy("num_automate", "date").orderBy("t")
w_5min = Window.partitionBy("num_automate").orderBy("t").rangeBetween(-300, 300)
w_heure_avant = Window.partitionBy("num_automate").orderBy("t").rangeBetween(-3600, -1)
transactions = (ops.filter(F.col("ts").isNotNull())
                .withColumn("t", F.unix_timestamp("ts"))
                .withColumn("delai_precedente_s", F.col("t") - F.lag("t").over(w_ordre))
                .withColumn("delai_suivante_s", F.lead("t").over(w_ordre) - F.col("t"))
                .withColumn("rang_dans_journee", F.row_number().over(w_jour_ordre))
                .withColumn("nb_ope_5min", F.count("*").over(w_5min))
                .withColumn("meme_montant_precedent",
                            ((F.col("est_retrait") == 1) & (F.col("montant") == F.lag("montant").over(w_ordre)))
                            .cast("int"))
                .withColumn("inabouti_precedent", F.coalesce(F.lag("inabouti").over(w_ordre), F.lit(0)))
                .withColumn("capture_dans_l_heure", F.sum("est_capture").over(w_heure_avant)))
if PERSISTER:
    transactions = transactions.persist(StorageLevel.MEMORY_AND_DISK)
print("Transactions enrichies :", fmt(transactions.count()))
exemple_terminal = transactions.select("num_automate").limit(1).collect()[0][0]
print("Exemple : les 25 premières opérations d'un terminal")
display(en_pandas(transactions.filter(F.col("num_automate") == exemple_terminal).orderBy("t").limit(25)
                  .select("num_automate", "ts", "type_op", "cat_reseau", "montant", "montant_billets", "inabouti",
                          "delai_precedente_s", "nb_ope_5min", "meme_montant_precedent", "capture_dans_l_heure")))

if DATASET_SORTIE_TRANSACTIONS:
    import dataiku.spark as dkuspark
    dkuspark.write_with_schema(dataiku.Dataset(DATASET_SORTIE_TRANSACTIONS), transactions.drop("t"))
    print("Table enrichie enregistrée dans le dataset", DATASET_SORTIE_TRANSACTIONS)

### Indicateurs par réseau, comptés transaction par transaction

In [ ]:
par_reseau = en_pandas(ops.groupBy("groupe_reseau", "cat_reseau").agg(
    F.sum("est_retrait").alias("retraits"),
    F.sum(F.when(F.col("type_op") == "RETRAIT", 1).otherwise(0)).alias("retraits_interne"),
    F.sum(F.when(F.col("type_op") == "AUTORISATION_RETRAIT", 1).otherwise(0)).alias("retraits_hors_interne"),
    F.sum(F.when(F.col("est_retrait") == 1, F.col("montant")).otherwise(0)).alias("montant"),
    F.sum("inabouti").alias("inaboutis"),
    F.sum("est_capture").alias("captures"),
    F.countDistinct("num_automate").alias("terminal")).orderBy(F.desc("retraits")))
par_reseau["pct_retraits"] = (100 * par_reseau["retraits"] / max(par_reseau["retraits"].sum(), 1)).round(3)
par_reseau["montant_moyen"] = (par_reseau["montant"] / par_reseau["retraits"].replace(0, np.nan)).round(2)
par_reseau["taux_inaboutis_pct"] = (100 * par_reseau["inaboutis"] / par_reseau["retraits"].replace(0, np.nan)).round(3)
display(par_reseau)

In [ ]:
JOURS_FR = {2: "lun", 3: "mar", 4: "mer", 5: "jeu", 6: "ven", 7: "sam", 1: "dim"}
ORDRE_JOURS = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]

# Vue de comptage terminal x jour, calculée à partir des transactions, uniquement pour les graphiques et les
# repérages par journée (les transactions restent la donnée de référence)
gj = (ops.groupBy("num_automate", "date")
      .agg(F.sum("est_retrait").alias("retraits"),
           F.sum("est_capture").alias("captures"),
           F.sum("est_capture_interne").alias("captures_interne"),
           F.sum("est_capture_hors_interne").alias("captures_hors_interne"),
           F.sum("inabouti").alias("inaboutis"),
           F.sum(F.when(F.col("est_retrait") == 1, F.col("est_etranger")).otherwise(0)).alias("etrangers"),
           F.sum(F.when(F.col("est_retrait") == 1, F.col("montant")).otherwise(0)).alias("montant"),
           F.sum(F.when(F.col("est_retrait") == 1, F.col("est_nuit")).otherwise(0)).alias("retraits_nuit"))
      .persist(StorageLevel.MEMORY_AND_DISK))
print("Lignes terminal x jour (jours avec au moins une opération) :", fmt(gj.count()))

## 5. Volumes dans le temps
Ce qu'on cherche : tendance de fond sur 3 ans, saisonnalité annuelle (Noël, été), hebdomadaire et horaire, effet des jours fériés, journées exceptionnelles à l'échelle du parc (incident réseau, panne générale).

In [ ]:
jour = en_pandas(ops.groupBy("date").agg(F.sum("est_retrait").alias("retraits"),
                                        F.sum("est_capture").alias("captures"),
                                        F.sum("inabouti").alias("inaboutis"),
                                        F.countDistinct("num_automate").alias("terminaux_actifs")))
jour["date"] = pd.to_datetime(jour["date"])
jour = jour.sort_values("date").set_index("date")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(jour.index, jour["retraits"], lw=0.6, alpha=0.5, label="par jour")
axes[0].plot(jour.index, jour["retraits"].rolling(7, center=True).mean(), lw=1.6, label="moyenne 7 jours")
axes[0].plot(jour.index, jour["retraits"].rolling(28, center=True).mean(), lw=1.6, label="moyenne 28 jours")
axes[0].set_title("Retraits par jour, parc complet")
axes[0].legend()
axes[1].plot(jour.index, jour["terminaux_actifs"], lw=0.8, color="tab:green")
axes[1].set_title("Nombre de terminaux ayant au moins une opération dans la journée")
plt.tight_layout()
plt.show()

In [ ]:
# Jours fériés et journées extrêmes du parc
try:
    import holidays
    feries = holidays.France(years=sorted(jour.index.year.unique()))
    jour["ferie"] = [d.date() in feries for d in jour.index]
except Exception:
    jour["ferie"] = False
    print("Librairie holidays indisponible dans ce kernel : jours fériés non marqués.")
jour["jour_semaine"] = jour.index.dayofweek.map(dict(enumerate(ORDRE_JOURS)))
attendu = jour.groupby("jour_semaine")["retraits"].transform(lambda s: s.rolling(9, center=True, min_periods=3).median())
jour["ratio_vs_meme_jour"] = (jour["retraits"] / attendu).round(2)
print("Journées du parc les plus éloignées de la normale du même jour de semaine :")
display(jour.reindex(jour["ratio_vs_meme_jour"].sub(1).abs().sort_values(ascending=False).index)
        .head(20)[["jour_semaine", "ferie", "retraits", "ratio_vs_meme_jour", "terminaux_actifs", "captures", "inaboutis"]])

In [ ]:
# Saisonnalité annuelle : retraits moyens par jour, par mois et par année
m = jour.groupby([jour.index.year, jour.index.month])["retraits"].mean().unstack(0)
m.plot(marker="o", title="Retraits moyens par jour, selon le mois (une courbe par année)")
plt.xlabel("mois")
plt.show()

In [ ]:
# Rythme hebdomadaire x horaire : retraits moyens par jour, pour chaque jour de semaine et heure
hh = en_pandas(ops.filter(F.col("est_retrait") == 1).groupBy("jour_semaine", "heure").count())
nb_par_jour_semaine = jour.groupby("jour_semaine").size()
hh["jour"] = hh["jour_semaine"].map(JOURS_FR)
hh["moyenne"] = hh["count"] / hh["jour"].map(nb_par_jour_semaine)
pivot = hh.pivot(index="jour", columns="heure", values="moyenne").reindex(ORDRE_JOURS)
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
ax.set_yticks(range(7))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_xlabel("heure")
ax.set_title("Retraits moyens par jour, selon le jour de la semaine et l'heure (parc complet)")
plt.colorbar(im)
plt.show()

## 6. Le parc de terminaux
Ce qu'on cherche : nombre de terminaux, entrées et sorties du parc, concentration de l'activité, terminal à activité intermittente.

In [ ]:
par_terminal = en_pandas(ops.groupBy("num_automate").agg(
    F.min("date").alias("premier_jour"), F.max("date").alias("dernier_jour"),
    F.countDistinct("date").alias("jours_actifs"),
    F.sum("est_retrait").alias("retraits"), F.sum("est_capture").alias("captures"),
    F.sum("inabouti").alias("inaboutis"),
    F.sum(F.when(F.col("est_retrait") == 1, F.col("est_etranger")).otherwise(0)).alias("etrangers"),
    F.sum(F.when(F.col("est_retrait") == 1, F.col("est_nuit")).otherwise(0)).alias("retraits_nuit"),
    F.sum(F.when(F.col("est_retrait") == 1, F.col("montant")).otherwise(0)).alias("montant")), max_lignes=100000)
par_terminal["premier_jour"] = pd.to_datetime(par_terminal["premier_jour"])
par_terminal["dernier_jour"] = pd.to_datetime(par_terminal["dernier_jour"])
par_terminal["duree_jours"] = (par_terminal["dernier_jour"] - par_terminal["premier_jour"]).dt.days + 1
par_terminal["taux_presence"] = par_terminal["jours_actifs"] / par_terminal["duree_jours"]
par_terminal["retraits_par_jour_actif"] = par_terminal["retraits"] / par_terminal["jours_actifs"]
par_terminal["taux_inaboutis_pct"] = 100 * par_terminal["inaboutis"] / par_terminal["retraits"].clip(lower=1)
par_terminal["captures_pour_1000"] = 1000 * par_terminal["captures"] / par_terminal["retraits"].clip(lower=1)
par_terminal["part_etrangers_pct"] = 100 * par_terminal["etrangers"] / par_terminal["retraits"].clip(lower=1)
par_terminal["part_nuit_pct"] = 100 * par_terminal["retraits_nuit"] / par_terminal["retraits"].clip(lower=1)
par_terminal["montant_moyen"] = par_terminal["montant"] / par_terminal["retraits"].clip(lower=1)
print("terminaux distincts :", fmt(len(par_terminal)))
display(par_terminal.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(np.log10(par_terminal["retraits_par_jour_actif"].clip(lower=0.1)), bins=60)
axes[0].set_title("Retraits par jour actif (log10)")
s = np.sort(par_terminal["retraits"].values)[::-1]
axes[1].plot(np.arange(1, len(s) + 1) / len(s) * 100, np.cumsum(s) / s.sum() * 100)
axes[1].set_title("Concentration : % des retraits réalisés par les x % de terminaux les plus actifs")
axes[1].set_xlabel("% des terminaux")
axes[2].hist(par_terminal["taux_presence"], bins=50)
axes[2].set_title("Taux de présence (jours actifs / durée de vie)")
plt.tight_layout()
plt.show()

entrees = par_terminal.groupby(par_terminal["premier_jour"].dt.to_period("M")).size()
sorties = par_terminal.groupby(par_terminal["dernier_jour"].dt.to_period("M")).size()
mouvements = pd.DataFrame({"premiere_operation": entrees, "derniere_operation": sorties}).fillna(0).astype(int)
print("Terminaux par mois de première et de dernière opération (le premier et le dernier mois reflètent les bornes de la table) :")
display(mouvements.T)
print("terminaux actifs moins de 50 % des jours de leur durée de vie :", fmt((par_terminal["taux_presence"] < 0.5).sum()))

## 6 bis. Géographie du parc
Qualité des coordonnées, puis cartes du parc : volume, part de cartes étrangères, taux d'inaboutis, captures. Sans fond de carte (pas d'accès Internet) : la densité des terminaux dessine d'elle-même la France.

In [ ]:
if not GEO_PRESENTE:
    print("Longitude / latitude absentes : section ignorée.")
else:
    ref_terminal = en_pandas(brut.groupBy("num_automate").agg(
        F.first(en_nombre("longitude"), True).alias("longitude"),
        F.first(en_nombre("latitude"), True).alias("latitude"),
        F.countDistinct(F.concat_ws("/", F.col("longitude").cast("string"), F.col("latitude").cast("string")))
        .alias("nb_coordonnees"),
        F.first("code_postal", True).alias("code_postal"), F.first("ville", True).alias("ville"),
        F.first("code_site", True).alias("code_site"), F.first("libelle_site", True).alias("libelle_site")),
        max_lignes=100000)
    par_terminal = par_terminal.merge(ref_terminal, on="num_automate", how="left")
    lon, lat = par_terminal["longitude"], par_terminal["latitude"]
    metropole = lon.between(-5.5, 10) & lat.between(41, 51.5)
    controles_geo = {
        "Terminal sans coordonnées": int((lon.isna() | lat.isna()).sum()),
        "coordonnées à 0": int(((lon == 0) | (lat == 0)).sum()),
        "hors métropole (DOM-TOM ou erreur)": int((lon.notna() & lat.notna() & ~metropole).sum()),
        "Terminaux avec plusieurs coordonnées": int((par_terminal["nb_coordonnees"] > 1).sum()),
        "coordonnées partagées par plusieurs terminaux": int(par_terminal[metropole].duplicated(["longitude", "latitude"],
                                                                                     keep=False).sum()),
    }
    display(pd.Series(controles_geo).to_frame("nb_terminaux"))
    m = par_terminal[metropole].copy()
    m["retraits_par_jour"] = m["retraits_par_jour_actif"]
    vues = [("retraits_par_jour", "Retraits par jour actif (log)", "viridis", True),
            ("part_etrangers_pct", "% de cartes étrangères", "magma_r", False),
            ("taux_inaboutis_pct", "% de retraits inaboutis", "Reds", False),
            ("captures_pour_1000", "Captures pour 1 000 retraits", "Purples", False)]
    fig, axes = plt.subplots(2, 2, figsize=(16, 15))
    for ax, (col, titre, cmap, log) in zip(axes.ravel(), vues):
        v = m[col].clip(upper=m[col].quantile(0.99))
        v = np.log10(v.clip(lower=0.1)) if log else v
        ordre = v.sort_values().index
        sc = ax.scatter(m.loc[ordre, "longitude"], m.loc[ordre, "latitude"], c=v[ordre], s=6, cmap=cmap, alpha=0.8)
        ax.set_aspect(1 / np.cos(np.radians(46.5)))
        ax.set_title(titre)
        plt.colorbar(sc, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()

    print("Villes les plus actives :")
    display(par_terminal.groupby("ville").agg(terminal=("num_automate", "size"), retraits=("retraits", "sum"),
                                         part_etrangers_pct=("part_etrangers_pct", "median"))
            .sort_values("retraits", ascending=False).head(20).round(2))
    print("Par type de site d'implantation :")
    display(par_terminal.groupby("libelle_site").agg(terminal=("num_automate", "size"),
                                                retraits_par_jour=("retraits_par_jour_actif", "median"),
                                                part_etrangers_pct=("part_etrangers_pct", "median"),
                                                part_nuit_pct=("part_nuit_pct", "median"),
                                                taux_inaboutis_pct=("taux_inaboutis_pct", "median"))
            .sort_values("terminal", ascending=False).head(30).round(2))

## 7. Montants
Ce qu'on cherche : distribution, montants les plus fréquents, plafonds, montants incompatibles avec les coupures, écarts entre montant demandé et billets délivrés.

In [ ]:
retraits = ops.filter(F.col("est_retrait") == 1)
q = retraits.select(F.expr("percentile_approx(montant, array(0.01,0.05,0.25,0.5,0.75,0.95,0.99,0.999), 10000)")
                    .alias("q")).collect()[0]["q"]
print("Quantiles du montant (1 %, 5 %, 25 %, 50 %, 75 %, 95 %, 99 %, 99,9 %) :", q)

h = en_pandas(retraits.filter(F.col("montant") <= 1500)
              .groupBy((F.floor(F.col("montant") / 10) * 10).alias("tranche")).count().orderBy("tranche"))
plt.bar(h["tranche"], h["count"], width=9)
plt.yscale("log")
plt.title("Distribution des montants retirés (tranches de 10 €, échelle log)")
plt.show()

top_montants = en_pandas(retraits.groupBy("montant").count().orderBy(F.desc("count")).limit(25))
top_montants["pct"] = (100 * top_montants["count"] / resume["retraits"]).round(2)
display(top_montants)

r = retraits.agg(
    F.sum((F.round(F.col("montant") * 100) % 500 != 0).cast("int")).alias("non_multiples_de_5"),
    F.sum((F.abs(F.col("ecart_billets")) > 0.001).cast("int")).alias("ecart_avec_billets"),
    F.sum((F.col("montant_billets") == 0).cast("int")).alias("aucun_billet_renseigne"),
).collect()[0].asDict()
for k, v in r.items():
    print("{:<26}: {} ({:.3f} % des retraits)".format(k, fmt(v), 100 * v / resume["retraits"]))

ecarts = en_pandas(retraits.filter(F.abs(F.col("ecart_billets")) > 0.001)
                   .groupBy(F.round("ecart_billets", 0).alias("ecart")).count().orderBy(F.desc("count")).limit(20))
print("Écarts (montant demandé - billets délivrés) les plus fréquents :")
display(ecarts)

In [ ]:
mr = en_pandas(retraits.groupBy("groupe_reseau").agg(
    F.count("*").alias("retraits"),
    F.expr("percentile_approx(montant, 0.5)").alias("montant_median"),
    F.avg("montant").alias("montant_moyen")).orderBy(F.desc("retraits")))
display(mr.round(1))
mh = en_pandas(retraits.groupBy("heure").agg(F.avg("montant").alias("montant_moyen"), F.count("*").alias("n")).orderBy("heure"))
mh.plot(x="heure", y="montant_moyen", marker="o", legend=False, title="Montant moyen selon l'heure")
plt.show()

## 8. Réseaux de cartes
Ce qu'on cherche : poids de chaque réseau, saisonnalité des cartes étrangères (tourisme), concentration des cartes étrangères sur quelques terminaux.

In [ ]:
res = en_pandas(retraits.groupBy("groupe_reseau", "cat_reseau").agg(F.count("*").alias("retraits"),
                                                                    F.sum("montant").alias("montant"))
                .orderBy(F.desc("retraits")))
res["pct_retraits"] = (100 * res["retraits"] / res["retraits"].sum()).round(3)
display(res)

rm = en_pandas(retraits.groupBy("mois", "groupe_reseau").count())
rm = rm.pivot(index="mois", columns="groupe_reseau", values="count").fillna(0).sort_index()
part = 100 * rm.div(rm.sum(axis=1), axis=0)
part.drop(columns=[part.sum().idxmax()]).plot(title="Part mensuelle des réseaux (hors réseau principal), en %")
plt.show()

rh = en_pandas(retraits.groupBy("groupe_reseau", "heure").count())
rh = rh.pivot(index="heure", columns="groupe_reseau", values="count").fillna(0)
(rh / rh.sum()).plot(title="Profil horaire de chaque réseau (part des retraits du réseau par heure)")
plt.show()

In [ ]:
g = par_terminal[par_terminal["retraits"] >= 1000].sort_values("part_etrangers_pct", ascending=False)
etr_total = par_terminal["etrangers"].sum()
for p in [1, 5, 10]:
    k = max(1, int(len(par_terminal) * p / 100))
    print("Les {} % de terminaux qui ont le plus de cartes étrangères concentrent {:.1f} % des retraits étrangers".format(
        p, 100 * par_terminal["etrangers"].nlargest(k).sum() / max(etr_total, 1)))
plt.hist(g["part_etrangers_pct"], bins=80)
plt.yscale("log")
plt.title("Part de cartes étrangères par terminal, en % des retraits")
plt.show()
display(g.head(20)[["num_automate", "retraits", "etrangers", "part_etrangers_pct", "retraits_par_jour_actif"]].round(2))

## 9. Retraits inaboutis
Deux causes distinguées : `top_annule = 1` et écart entre montant demandé et billets délivrés.

In [ ]:
c = retraits.agg(F.sum("inabouti").alias("inaboutis"), F.sum("inabouti_annule").alias("annules"),
                 F.sum("inabouti_billets").alias("ecart_billets"),
                 F.sum((F.col("inabouti_annule") * F.col("inabouti_billets"))).alias("les_deux")).collect()[0].asDict()
for k, v in c.items():
    print("{:<14}: {} ({:.3f} % des retraits)".format(k, fmt(v), 100 * v / resume["retraits"]))

ih = en_pandas(retraits.groupBy("heure").agg((100 * F.avg("inabouti")).alias("taux_pct")).orderBy("heure"))
ir = en_pandas(retraits.groupBy("groupe_reseau").agg((100 * F.avg("inabouti")).alias("taux_pct"),
                                                     F.count("*").alias("n")).orderBy(F.desc("taux_pct")))
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(ih["heure"], ih["taux_pct"], marker="o")
axes[0].set_title("Taux d'inaboutis selon l'heure (%)")
axes[1].barh(ir["groupe_reseau"], ir["taux_pct"])
axes[1].set_title("Taux d'inaboutis selon le réseau (%)")
plt.tight_layout()
plt.show()

g = par_terminal[par_terminal["retraits"] >= 500]
plt.hist(g["taux_inaboutis_pct"], bins=80)
plt.yscale("log")
plt.title("Taux d'inaboutis par terminal (terminal avec au moins 500 retraits)")
plt.show()
display(g.nlargest(20, "taux_inaboutis_pct")[["num_automate", "retraits", "inaboutis", "taux_inaboutis_pct"]].round(2))

In [ ]:
# Journées avec un taux d'inaboutis élevé
gj_inab = gj.filter(F.col("retraits") >= 5).withColumn("taux", F.col("inaboutis") / F.col("retraits"))
dist = en_pandas(gj_inab.groupBy(F.least(F.col("inaboutis"), F.lit(30)).alias("inaboutis_dans_la_journee")).count()
                 .orderBy("inaboutis_dans_la_journee"))
display(dist.set_index("inaboutis_dans_la_journee").T)
print("Journées avec >= 5 retraits et > 20 % d'inaboutis :", fmt(gj_inab.filter(F.col("taux") > 0.2).count()))

## 10. Captures de carte

In [ ]:
captures = ops.filter(F.col("est_capture") == 1)
c = ops.agg(F.sum("est_capture_interne").alias("captures_interne"), F.sum("est_capture_hors_interne").alias("captures_hors_interne"),
            F.sum(F.when(F.col("type_op") == "RETRAIT", 1).otherwise(0)).alias("retraits_interne"),
            F.sum(F.when(F.col("type_op") == "AUTORISATION_RETRAIT", 1).otherwise(0)).alias("retraits_hors_interne")
            ).collect()[0].asDict()
print("Captures réseau interne : {} | pour 1 000 retraits réseau interne : {:.2f}".format(
    fmt(c["captures_interne"]), 1000 * c["captures_interne"] / max(c["retraits_interne"], 1)))
print("Captures hors réseau interne             : {} | pour 1 000 retraits hors réseau interne : {:.2f}".format(
    fmt(c["captures_hors_interne"]), 1000 * c["captures_hors_interne"] / max(c["retraits_hors_interne"], 1)))
cm = en_pandas(captures.groupBy("motif_capture", F.when(F.col("est_reseau_interne") == 1, "réseau interne")
                                .otherwise("hors réseau interne").alias("reseau")).count().orderBy(F.desc("count")))
display(cm.pivot(index="motif_capture", columns="reseau", values="count").fillna(0).astype(int))
print("Captures par réseau de la carte :")
display(en_pandas(captures.groupBy("cat_reseau").count().orderBy(F.desc("count")).limit(25)))
ch = en_pandas(captures.groupBy("heure").count().orderBy("heure"))
ch.plot(x="heure", y="count", marker="o", legend=False, title="Captures selon l'heure")
plt.show()

g = par_terminal[par_terminal["retraits"] >= 500]
display(g.nlargest(20, "captures_pour_1000")[["num_automate", "retraits", "captures", "captures_pour_1000"]].round(2))
dist = en_pandas(gj.groupBy(F.least(F.col("captures"), F.lit(15)).alias("captures_dans_la_journee")).count()
                 .orderBy("captures_dans_la_journee"))
display(dist.set_index("captures_dans_la_journee").T)

## 10 bis. Date de validité des cartes
Format détecté automatiquement (MMAA ou AAMM sur 4 chiffres, ou date complète). On mesure la part de retraits faits avec une carte déjà expirée, et le délai restant avant expiration.

In [ ]:
val = ops.filter(F.col("est_retrait") == 1).select("num_automate", "date",
                                                     F.regexp_replace("validite_txt", "[^0-9]", "").alias("v"))
apercu = en_pandas(val.groupBy(F.length("v").alias("longueur")).count().orderBy(F.desc("count")))
print("Longueur des dates de validité (en chiffres) :")
display(apercu)
longueurs = apercu.dropna()
longueurs = longueurs[longueurs["longueur"] > 0]
longueur = int(longueurs["longueur"].iloc[0]) if len(longueurs) else 0
if longueur == 4:
    t = val.filter(F.length("v") == 4).agg(
        F.avg((F.substring("v", 1, 2).cast("int") <= 12).cast("int")).alias("mois_devant"),
        F.avg((F.substring("v", 3, 2).cast("int") <= 12).cast("int")).alias("mois_derriere")).collect()[0]
    mois_devant = t["mois_devant"] >= t["mois_derriere"]
    print("Format retenu :", "MMAA" if mois_devant else "AAMM")
    mm = F.substring("v", 1, 2) if mois_devant else F.substring("v", 3, 2)
    aa = F.substring("v", 3, 2) if mois_devant else F.substring("v", 1, 2)
    fin_validite = F.last_day(F.to_date(F.concat(F.lit("20"), aa, F.lit("-"), mm, F.lit("-01"))))
elif longueur in (6, 8):
    fmt_v = "yyyyMM" if longueur == 6 else "yyyyMMdd"
    fin_validite = F.last_day(F.to_date("v", fmt_v))
else:
    fin_validite = None
    print("Format de date de validité non reconnu : analyse ignorée.")
if fin_validite is not None:
    val = val.withColumn("mois_restants", F.months_between(fin_validite, F.col("date")))
    r = val.agg(F.avg((F.col("mois_restants") < 0).cast("int")).alias("part_expirees"),
                F.avg(F.col("mois_restants").isNull().cast("int")).alias("part_illisibles")).collect()[0]
    print("Retraits avec une carte expirée : {:.3f} % | dates illisibles : {:.3f} %".format(
        100 * (r["part_expirees"] or 0), 100 * (r["part_illisibles"] or 0)))
    h = en_pandas(val.groupBy(F.floor("mois_restants").alias("mois")).count().orderBy("mois"))
    h = h[h["mois"].between(-12, 72)]
    plt.bar(h["mois"], h["count"], width=0.9)
    plt.title("Mois restants avant expiration de la carte au moment du retrait")
    plt.show()
    exp_terminal = en_pandas(val.groupBy("num_automate").agg(F.count("*").alias("retraits"),
                                                          (100 * F.avg((F.col("mois_restants") < 0).cast("int")))
                                                          .alias("pct_cartes_expirees"))
                        .filter(F.col("retraits") >= 200).orderBy(F.desc("pct_cartes_expirees")).limit(20))
    print("Terminal où les cartes expirées sont les plus fréquentes :")
    display(exp_terminal.round(3))

## 11. Dynamique à l'intérieur de la journée
C'est l'information que la fiche d'identité (une ligne par terminal et par jour) ne voit pas. Étudiée sur les **N derniers jours** (paramètre `JOURS_INTRAJOURNEE`).

- **Délai entre deux opérations successives** d'un même terminal.
- **Rafales** : opérations à moins de 30 s d'intervalle, et répétitions du même montant à moins de 2 min.
- **Silences** : plus longue période sans opération entre 7 h et 22 h, comparée au rythme habituel du terminal. Avec un rythme moyen de *r* opérations par heure, la probabilité qu'un silence de *d* heures arrive par hasard vaut environ exp(−*r*·*d*) (approximation de Poisson à rythme constant).

In [ ]:
debut_intra = F.date_sub(F.lit(str(DATE_MAX)).cast("date"), JOURS_INTRAJOURNEE)
w_terminal = Window.partitionBy("num_automate").orderBy("ts")
intra = (ops.filter((F.col("date") > debut_intra) & (F.col("est_retrait") == 1) & F.col("ts").isNotNull())
         .withColumn("t", F.unix_timestamp("ts"))
         .withColumn("delta_s", F.col("t") - F.lag("t").over(w_terminal))
         .withColumn("meme_montant_que_precedent", (F.col("montant") == F.lag("montant").over(w_terminal)).cast("int"))
         .persist(StorageLevel.MEMORY_AND_DISK))
print("Retraits étudiés :", fmt(intra.count()))

tranches = (F.when(F.col("delta_s") < 10, "1. < 10 s").when(F.col("delta_s") < 30, "2. 10-30 s")
            .when(F.col("delta_s") < 60, "3. 30-60 s").when(F.col("delta_s") < 300, "4. 1-5 min")
            .when(F.col("delta_s") < 900, "5. 5-15 min").when(F.col("delta_s") < 3600, "6. 15-60 min")
            .when(F.col("delta_s") < 3 * 3600, "7. 1-3 h").when(F.col("delta_s") < 12 * 3600, "8. 3-12 h")
            .otherwise("9. >= 12 h"))
dd = en_pandas(intra.filter(F.col("delta_s").isNotNull()).groupBy(tranches.alias("delai")).count().orderBy("delai"))
dd["pct"] = (100 * dd["count"] / dd["count"].sum()).round(2)
display(dd)

In [ ]:
# Rafales par journée terminal
rafales = (intra.groupBy("num_automate", "date")
           .agg(F.count("*").alias("retraits"),
                F.sum((F.col("delta_s") < 30).cast("int")).alias("enchainements_moins_30s"),
                F.sum(((F.col("delta_s") < 120) & (F.col("meme_montant_que_precedent") == 1)).cast("int"))
                .alias("meme_montant_moins_2min")))
print("Journées avec le plus d'enchaînements à moins de 30 s :")
display(en_pandas(rafales.orderBy(F.desc("enchainements_moins_30s")).limit(20)))
print("Journées avec le plus de répétitions du même montant à moins de 2 min :")
display(en_pandas(rafales.orderBy(F.desc("meme_montant_moins_2min")).limit(20)))

In [ ]:
# Silences de jour
jour_ops = intra.filter((F.col("heure") >= HEURE_DEBUT_JOUR) & (F.col("heure") < HEURE_FIN_JOUR))
w_jour = Window.partitionBy("num_automate", "date").orderBy("ts")
t0 = F.unix_timestamp(F.concat(F.col("date").cast("string"), F.lit(" {:02d}:00:00".format(HEURE_DEBUT_JOUR))))
t1 = F.unix_timestamp(F.concat(F.col("date").cast("string"), F.lit(" {:02d}:00:00".format(HEURE_FIN_JOUR))))
sil = (jour_ops.withColumn("gap_s", F.col("t") - F.lag("t").over(w_jour))
       .groupBy("num_automate", "date")
       .agg(F.count("*").alias("retraits_jour"), F.max("gap_s").alias("gap_interne_s"),
            F.min("t").alias("premier_t"), F.max("t").alias("dernier_t"),
            F.first(t0).alias("t0"), F.first(t1).alias("t1"))
       .withColumn("silence_h", F.greatest(F.coalesce(F.col("gap_interne_s"), F.lit(0)),
                                           F.col("premier_t") - F.col("t0"),
                                           F.col("t1") - F.col("dernier_t")) / 3600.0))
rythme = (sil.groupBy("num_automate")
          .agg(F.expr("percentile_approx(retraits_jour, 0.5)").alias("retraits_jour_median"))
          .withColumn("rythme_par_heure", F.col("retraits_jour_median") / float(HEURE_FIN_JOUR - HEURE_DEBUT_JOUR)))
sil = (sil.join(rythme, "num_automate")
       .withColumn("retraits_attendus_pendant_silence", F.col("rythme_par_heure") * F.col("silence_h"))
       .withColumn("proba_hasard", F.exp(-F.col("retraits_attendus_pendant_silence")))
       .persist(StorageLevel.MEMORY_AND_DISK))

dist_sil = en_pandas(sil.groupBy(F.least(F.floor("silence_h"), F.lit(15)).alias("silence_max_heures")).count()
                     .orderBy("silence_max_heures"))
display(dist_sil.set_index("silence_max_heures").T)
print("Journées avec un silence improbable (moins de 1 chance sur 1000 par hasard) :",
      fmt(sil.filter(F.col("proba_hasard") < 1e-3).count()))
display(en_pandas(sil.orderBy("proba_hasard").limit(20)
                  .select("num_automate", "date", "retraits_jour", "retraits_jour_median", "silence_h",
                          "retraits_attendus_pendant_silence", "proba_hasard")))

par_jour_sil = en_pandas(sil.filter(F.col("proba_hasard") < 1e-3).groupBy("date").count().orderBy("date"))
par_jour_sil.plot(x="date", y="count", legend=False,
                  title="Nombre de terminaux en silence improbable, par jour (un pic = incident commun à plusieurs terminaux)")
plt.show()

In [ ]:
nuit = intra.agg((100 * F.avg("est_nuit")).alias("pct_nuit")).collect()[0]["pct_nuit"]
print("Part des retraits la nuit (22 h - 6 h) sur la période : {:.2f} %".format(nuit))

## 12. Afflux de cartes étrangères par rapport à l'habitude du terminal

In [ ]:
gj_recent = gj.filter(F.col("date") > debut_intra)
hab_etr = gj_recent.groupBy("num_automate").agg(F.avg("etrangers").alias("etrangers_moyen_par_jour"))
afflux = (gj_recent.join(hab_etr, "num_automate")
          .withColumn("ratio", F.col("etrangers") / F.greatest(F.col("etrangers_moyen_par_jour"), F.lit(0.2)))
          .filter(F.col("etrangers") >= 5))
display(en_pandas(afflux.orderBy(F.desc("ratio")).limit(20)
                  .select("num_automate", "date", "retraits", "etrangers", "etrangers_moyen_par_jour", "ratio")))

## 12 bis. Carte des terminaux atypiques et raison principale
Pour chaque terminal, sur la période étudiée, on compte les journées où chaque comportement atypique a été repéré par les sections précédentes. La raison principale est le comportement le plus fréquent. C'est une première version, purement descriptive, de la carte attendue par le métier.

| Comportement | Règle de repérage (journée) |
|---|---|
| silence | silence de jour avec moins de 1 chance sur 1 000 d'arriver par hasard |
| rafale | au moins 5 retraits du même montant à moins de 2 minutes d'intervalle |
| inaboutis | au moins 5 retraits et plus de 20 % d'inaboutis |
| captures | au moins 3 cartes capturées |
| cartes étrangères | au moins 5 retraits étrangers et 3 fois plus que d'habitude |

In [ ]:
gj_periode = gj.filter(F.col("date") > debut_intra)
comptes = {
    "silence": sil.filter(F.col("proba_hasard") < 1e-3).groupBy("num_automate").count(),
    "rafale": rafales.filter(F.col("meme_montant_moins_2min") >= 5).groupBy("num_automate").count(),
    "inaboutis": gj_periode.filter((F.col("retraits") >= 5) & (F.col("inaboutis") > 0.2 * F.col("retraits")))
                 .groupBy("num_automate").count(),
    "captures": gj_periode.filter(F.col("captures") >= 3).groupBy("num_automate").count(),
    "cartes étrangères": afflux.filter(F.col("ratio") >= 3).groupBy("num_automate").count(),
}
anomalies = None
for nom, sdf in comptes.items():
    p = en_pandas(sdf, max_lignes=100000).rename(columns={"count": nom}).set_index("num_automate")
    anomalies = p if anomalies is None else anomalies.join(p, how="outer")
anomalies = anomalies.fillna(0).astype(int)
anomalies["jours_atypiques"] = anomalies.sum(axis=1)
anomalies["raison_principale"] = anomalies[list(comptes)].idxmax(axis=1)


def phrase(ligne):
    morceaux = ["{} : {} j".format(n, int(ligne[n])) for n in comptes if ligne[n] > 0]
    return ", ".join(morceaux)


anomalies["detail"] = anomalies.apply(phrase, axis=1)
anomalies = anomalies.sort_values("jours_atypiques", ascending=False)
print("Terminaux avec au moins une journée atypique : {} sur {}".format(fmt(len(anomalies)), fmt(len(par_terminal))))
display(anomalies["raison_principale"].value_counts().to_frame("terminal"))
colonnes_geo = [c for c in ["longitude", "latitude", "ville", "libelle_site"] if c in par_terminal.columns]
top = anomalies.join(par_terminal.set_index("num_automate")[colonnes_geo + ["retraits_par_jour_actif"]], how="left")
display(top.head(30)[colonnes_geo[2:] + ["jours_atypiques", "raison_principale", "detail"]])

if GEO_PRESENTE:
    fond = par_terminal[par_terminal["longitude"].between(-5.5, 10) & par_terminal["latitude"].between(41, 51.5)]
    t = top[top["longitude"].between(-5.5, 10) & top["latitude"].between(41, 51.5)]
    couleurs = {"silence": "black", "rafale": "tab:red", "inaboutis": "tab:orange", "captures": "tab:purple",
                "cartes étrangères": "tab:green"}
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.scatter(fond["longitude"], fond["latitude"], s=3, color="#c8cdd6", label="Terminal sans anomalie marquée")
    for raison, c in couleurs.items():
        sel = t[t["raison_principale"] == raison]
        ax.scatter(sel["longitude"], sel["latitude"], s=8 + 4 * np.sqrt(sel["jours_atypiques"]), color=c,
                   alpha=0.75, label="{} ({} terminal)".format(raison, len(sel)))
    for g, l in t.head(10).iterrows():
        ax.annotate("{} · {}".format(g, l["raison_principale"]), (l["longitude"], l["latitude"]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_aspect(1 / np.cos(np.radians(46.5)))
    ax.set_title("terminaux atypiques sur la période : couleur = raison principale, taille = nombre de journées")
    ax.legend(loc="lower left", fontsize=9)
    plt.show()

## 13. Regarder une journée de près
Pour un terminal et une date repérés plus haut : opérations par quart d'heure ce jour-là, comparées à la médiane des 8 mêmes jours de semaine précédents.

In [ ]:
def regarder_journee(num_automate, date_txt):
    d = pd.Timestamp(date_txt)
    dates = [d - pd.Timedelta(weeks=k) for k in range(0, 9)]
    sub = (ops.filter((F.col("num_automate") == num_automate) &
                      F.col("date").cast("string").isin([x.strftime("%Y-%m-%d") for x in dates]))
           .groupBy("date", "type_op", ((F.col("heure") * 60 + F.col("minute")) / 15).cast("int").alias("quart"))
           .agg(F.count("*").alias("n"), F.sum("inabouti").alias("inaboutis")))
    p = en_pandas(sub)
    if p.empty:
        print("Aucune opération trouvée.")
        return
    p["date"] = pd.to_datetime(p["date"])
    ret = p[p["type_op"] != "CAPTURE"].groupby(["date", "quart"])["n"].sum().unstack(fill_value=0)
    ret = ret.reindex(columns=range(96), fill_value=0)
    jour_cible = ret.loc[d] if d in ret.index else pd.Series(0, index=range(96))
    reference = ret.drop(index=d, errors="ignore").median()
    x = np.arange(96) / 4
    plt.figure(figsize=(14, 4))
    plt.step(x, reference.values, where="post", label="médiane des 8 mêmes jours précédents", color="grey")
    plt.step(x, jour_cible.values, where="post", label=date_txt, color="tab:red")
    caps = p[(p["type_op"] == "CAPTURE") & (p["date"] == d)]
    if len(caps):
        plt.scatter(caps["quart"] / 4, [0] * len(caps), marker="x", s=80, color="black", label="captures")
    plt.title("Terminal {} : retraits par quart d'heure".format(num_automate))
    plt.xlabel("heure")
    plt.legend()
    plt.show()
    print("Ce jour : {} retraits ; médiane de référence : {:.0f}".format(int(jour_cible.sum()), reference.sum()))


# Exemple : remplacer par un terminal et une date repérés dans les sections 9 à 12
# regarder_journee("T00123", "2026-03-14")

## 13 bis. Voir les anomalies heure par heure
**Carte du parc jour x heure.** Chaque case compare les retraits du parc à cette heure-là à la médiane des mêmes jour de semaine et heure. Une case bleue isolée signale un incident commun à beaucoup de terminaux à cette heure (panne réseau, maintenance), invisible dans un total journalier.

**Carte d'un terminal jour x heure.** Une ligne par jour, une colonne par heure, sur toute une période. Un rectangle noir montre une panne (même partielle, de quelques heures), une case claire isolée une rafale. Les croix marquent les inaboutis et les captures.

In [ ]:
ph = en_pandas(ops.filter(F.col("est_retrait") == 1).groupBy("date", "heure").count())
ph["date"] = pd.to_datetime(ph["date"])
m = ph.pivot(index="date", columns="heure", values="count").reindex(columns=range(24)).fillna(0).sort_index()
ref = m.groupby(m.index.dayofweek).transform("median")
ratio = np.log2((m + 1) / (ref + 1)).clip(-2, 2)
fig, ax = plt.subplots(figsize=(10, 14))
im = ax.imshow(ratio.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
pos = [i for i, d in enumerate(ratio.index) if d.day == 1]
ax.set_yticks(pos)
ax.set_yticklabels([ratio.index[i].strftime("%m/%y") for i in pos], fontsize=7)
ax.set_xticks(range(24))
ax.set_xlabel("heure")
ax.set_title("Parc complet : retraits par heure / médiane habituelle (log2, bleu = déficit)")
plt.colorbar(im, label="log2(retraits / habitude)")
plt.show()
print("Créneaux (jour, heure) les plus déficitaires pour l'ensemble du parc :")
creneaux = ratio.stack().sort_values().head(15).rename("log2_ratio").reset_index()
display(creneaux)

In [ ]:
def carte_heures_terminal(num_automate, debut=None, fin=None):
    f = ops.filter(F.col("num_automate") == num_automate)
    if debut:
        f = f.filter(F.col("date") >= F.lit(debut).cast("date"))
    if fin:
        f = f.filter(F.col("date") <= F.lit(fin).cast("date"))
    p = en_pandas(f.groupBy("date", "heure").agg(F.sum("est_retrait").alias("retraits"),
                                                 F.sum("inabouti").alias("inaboutis"),
                                                 F.sum("est_capture").alias("captures"),
                                                 F.sum(F.when(F.col("est_retrait") == 1, F.col("est_etranger"))
                                                       .otherwise(0)).alias("etrangers")))
    if p.empty:
        print("Aucune opération.")
        return
    p["date"] = pd.to_datetime(p["date"])
    jours = pd.date_range(p["date"].min(), p["date"].max())
    r = p.pivot(index="date", columns="heure", values="retraits").reindex(index=jours, columns=range(24)).fillna(0)
    fig, ax = plt.subplots(figsize=(11, max(4, len(jours) * 0.04)))
    im = ax.imshow(np.log1p(r.values), aspect="auto", cmap="magma", interpolation="nearest")
    for col, marque, couleur in [("inaboutis", "x", "cyan"), ("captures", "o", "lime")]:
        s = p[p[col] > 0]
        y = [jours.get_loc(d) for d in s["date"]]
        ax.scatter(s["heure"], y, marker=marque, s=12, color=couleur, label=col)
    pos = [i for i, d in enumerate(jours) if d.day == 1 or i == 0]
    ax.set_yticks(pos)
    ax.set_yticklabels([jours[i].strftime("%d/%m/%y") for i in pos], fontsize=7)
    ax.set_xticks(range(24))
    ax.set_xlabel("heure")
    ax.set_title("Terminal {} : retraits par heure (log), un jour par ligne".format(num_automate))
    ax.legend(loc="upper right", fontsize=8)
    plt.colorbar(im, label="log(1 + retraits)")
    plt.show()


# Exemples : un terminal repéré en § 11 (silence) ou § 12 (afflux), sur une période de 2 à 6 mois
# carte_heures_terminal("T00123", "2026-01-01", "2026-06-30")

## 14. Synthèse des comportements observés
À compléter après exécution, pour chacun des comportements de l'introduction :

| Comportement | Existe dans les données ? | Fréquence (journées) | Exemples (terminal, date) | Visible dans la fiche journalière ? |
|---|---|---|---|---|
| Silence anormal | | | | |
| Rafale / même montant répété | | | | |
| Inaboutis en série | | | | |
| Captures anormales | | | | |
| Afflux de cartes étrangères | | | | |
| Cartes expirées | | | | |
| Autre comportement découvert | | | | |

Les comportements qui **ne sont pas visibles dans la fiche journalière** justifient le passage au grain opération.

In [ ]:
# Libère la mémoire du cluster en fin d'analyse
for t in [intra, sil, gj, ops]:  # tables gardées en mémoire du cluster
    try:
        t.unpersist()
    except Exception:
        pass